In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN, KMeans
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

glass = pd.read_csv("C:/Python/Cases/Glass Identification/Glass.csv")
X, y = glass.drop('Type', axis=1), glass['Type']
X.shape

scaler = StandardScaler().set_output(transform='pandas')
df_scaled = scaler.fit_transform(X)

eps_range = np.linspace(0.01, 1.5, 10)
mp_range = [2,3,4,5]
cnt = 0
a =[]
for i in eps_range:
    for j in mp_range:
        clust_DB = DBSCAN(eps=i, min_samples=j)
        clust_DB.fit(df_scaled.iloc[:,:9])
        if len(set(clust_DB.labels_)) > 2:
            cnt = cnt + 1
            df_scaled['Clust'] = clust_DB.labels_
            df_scl_inliers = df_scaled[df_scaled['Clust']!=-1]
            sil_sc = silhouette_score(df_scl_inliers.iloc[:,:-1], df_scl_inliers.iloc[:,-1])
            a.append([cnt,i,j,sil_sc])

pa = pd.DataFrame(a,columns=['Sr','eps','min_pt','sil'])
pa.sort_values('sil', ascending=False).iloc[0]

clust_DB = DBSCAN(eps=0.175556, min_samples=3)
clust_DB.fit(df_scaled.iloc[:,:9])
set(clust_DB.labels_)

np.unique(clust_DB.labels_, return_counts=True)

##### K-Means

df_scaled = scaler.fit_transform(X)

Ks = [2,3,4,5,6,7]
scores = []
for i in Ks:
    clust = KMeans(n_clusters=i, random_state=25)
    clust.fit(df_scaled)
    scores.append([i,silhouette_score(df_scaled, clust.labels_)])

df_scores = pd.DataFrame(scores, columns=['clusters','score'])
df_scores.sort_values('score', ascending=False)

clust = KMeans(n_clusters=2, random_state=25)
clust.fit(df_scaled)

clust.labels_

df_scaled.columns

prcomp = PCA(n_components=2).set_output(transform='pandas')
PC_Data = prcomp.fit_transform(df_scaled)

PC_Data['cluster'] = clust.labels_
PC_Data['cluster'] = PC_Data['cluster'].astype(str)

PC_Data['Type'] = y

sns.scatterplot(data=PC_Data, x='pca0', y='pca1', hue='cluster')
plt.show()

sns.scatterplot(data=PC_Data, x='pca0', y='pca1', hue='Type')
plt.show()